Parameters

In [ ]:
import os
import json
from pathlib import Path

parameters_filename = PARAMETERS_FILENAME
PROJECT_ROOT = Path("/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project")
parameters_path = PROJECT_ROOT / "parameters_jsons" / parameters_filename
with open(parameters_path, "r", encoding="utf-8") as file:
    parameters_dict = json.load(file)

In [ ]:
try:
    cuda_index = CUDA_IDX
    if cuda_index is None:
        raise NameError
except NameError:
    cuda_index = 0

# Imports

In [ ]:
from collections import Counter
from collections import defaultdict
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from openpyxl.utils.dataframe import dataframe_to_rows
from operator import sub
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import types
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import contextlib
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
from transformers import AutoModel, AutoTokenizer
import gc
import h5py  # For efficient storage of large arrays
import hashlib
import json
import matplotlib.pyplot as plt
import numpy as np
import time
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import openpyxl
import os
import pandas as pd
import pickle
import random
import re
import seaborn as sns
import string
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import shutil
import xlsxwriter
from scipy.sparse import csr_matrix
from captum.attr import IntegratedGradients, LayerConductance, LayerActivation
from captum.attr import visualization as viz

In [ ]:
# Set device globally
device = torch.device(f"cuda:{cuda_index}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    print(f"CUDA device #: {cuda_index}")
    print(f"GPU: {torch.cuda.get_device_name(cuda_index)}")
    print(f"CUDA Version: {torch.version.cuda}")

    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(False)


In [ ]:
def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
set_global_seed(parameters_dict['random_seed'])

In [ ]:
from config import get_model_config, COMPLETION_LABELS
from data.load_data import mount_drive, load_data
from data.process_data import append_prefix_to_proceeding_ids, filter_proceedings, remove_multi_defendant_cases, create_combined_dockets, filter_to_magistrate_started_cases, remove_extradition_cases
from data.sequence_builder import create_enhanced_training_sequences_cached, SequenceCacheBuilder
from regex_labels.label_generator_and_cache_manager import generate_and_cache_regex_labels
from models.transformer_creation import create_model_from_data
from training.trainer import train_case_transformer_with_mlb
from training.label_processor import EnhancedLabelProcessor
from training.evaluation import evaluate_with_teacher_forcing
from analysis.cooccurrence import export_cooccurrence_from_valid_completed_cases, create_cooccurrence_matrix_excel
from analysis.frequency_analysis import print_label_distribution_by_court
from analysis.diagnostic import comprehensive_training_diagnostic
from models.baseline_model import run_baseline, convert_baseline_results_to_table_format
from utils.cache_directory_utils import CacheDirectoryManager
from utils.model_memory_utils import analyze_model_memory, debug_oom_sample
from models.embeddings import extract_and_cache_bert_embeddings_streaming
from analysis.byclerk import analyze_clerk_data
from models.interpretability import CaseTransformerInterpretability, CaptumWrapper
from models.most_confident_wrong_answer_analyzer import run_confident_misclassification_analysis
from data.k_fold_methods import KFoldCVRunner
from data.dataset_creation import sample_from_the_longest_cases, export_long_cases_to_xlsx, sample_uniformly, export_uniform_cases_to_xlsx
from analysis.tablegenerator import TableGenerator
from data.balanced_dataset_creation import create_balanced_dataset, export_balanced_cases_to_xlsx
from models.export_model import export_model_complete
from utils.case_identifier_utils import extract_case_identifier_from_sequences, save_case_identifiers, load_case_identifiers, compare_case_identifiers, load_and_compare_case_identifiers
from models.baseline_model_2 import run_random_baseline_comparison, convert_random_baseline_results_to_table_format
from analysis.compare_transformer_to_baseline_1 import compare_transformer_vs_baseline1_on_completion_labels, convert_baseline1_comparison_to_table_generator_format
# ═══════════════════════════════════════════════════════════════
# Embedding Clustering Analysis (PCA + KMeans on pre-classifier vectors)
# ═══════════════════════════════════════════════════════════════

from analysis.pca_vectors import (run_embedding_clusters_pipeline, run_embedding_clusters_kfolds_pipeline)


In [ ]:
%load_ext autoreload
%autoreload 2
from notebook_helpers import (
    # Data filtering helpers
    filter_to_completed_cases,
    build_case_counts_df,
    filter_zero_label_proceedings,
    generate_cooccurrence_matrix_excel,
    filter_proceedings_to_end_of_case,
    apply_skipped_duplicates_on_imported_sequences,
    
    # Data transformation helpers
    transform_labels_to_binary,
    convert_data_to_format,
    
    # Sequence processing helpers
    filter_sequences_with_meaningful_labels,
    split_train_validation,
    prepare_sequences_and_labels,
    
    # Case info extraction
    extract_case_info_from_sequence,
    
    # Prediction helpers
    predict_case_labels,
    predict_step,
    
    # Metrics calculation helpers
    calculate_metrics,
    calculate_case_metrics,
    collect_labels_from_binary,
    count_processed_steps,
    
    # Analysis helpers
    perform_frequency_analysis,
    debug_model_sample,
    
    # Training helpers
    train_model,
    
    # Evaluation helpers
    prepare_validation_data_for_eval,
    evaluate_model,
    print_micro_averaged_metrics,
    print_top_performing_labels,
    
    # Case-level evaluation helpers
    evaluate_case_level,
    print_case_level_metrics,
    save_case_level_results,
    print_best_worst_cases,
    print_f1_distribution,
    calculate_label_metrics_from_cases,
    save_label_metrics_to_excel,
    
    # Advanced analysis helpers
    run_diagnostic_analysis,
    run_baseline_comparison,
    run_confident_misclassification,
    
    # Interpretability helpers
    initialize_interpretability_analyzer,
    select_samples_for_analysis,
    get_model_prediction,
    compute_attributions,
    visualize_attributions,
    get_and_save_feature_summary,
    analyze_single_sample,
    run_attribution_analysis,
    aggregate_component_importance,
    save_and_visualize_aggregate_importance,
    find_sample_with_label,
    analyze_label_specific_features,
    create_label_feature_comparison,
    run_interpretability_analysis,
    analyze_confident_misclass_for_label,
    compute_global_attributions_table,
    
    # Sample prediction helpers
    print_step_prediction_analysis,
    analyze_sample_predictions,

    # ablation study helpers
    create_standard_ablation_configurations,
    run_ablation_study,
    compare_ablation_table
)

In [ ]:
folder = rf'/data/krosen5/cache/transformers/{parameters_dict["folder_name_directory"]}/'
if not os.path.exists(folder):
    os.makedirs(folder)
model_cache_manager = CacheDirectoryManager(base_path=folder)

In [ ]:
if parameters_dict['sequence_data_mode'] == 'import' and parameters_dict.get('sequence_data_base_directory'):
    base_folder = rf'/data/krosen5/cache/transformers/{parameters_dict["sequence_data_base_directory"]}/'
    import_cache_manager = CacheDirectoryManager(base_path=base_folder)
elif parameters_dict['sequence_data_mode'] == 'import':
    import_cache_manager = model_cache_manager

In [ ]:
if parameters_dict['regex_labels_mode'] == 'load_in' and parameters_dict.get('regex_labels_load_in_folder'):
    regex_folder = rf'/data/krosen5/cache/transformers/{parameters_dict["regex_labels_load_in_folder"]}/'
    #make it into a path
    regex_folder = Path(regex_folder)

In [ ]:
table_generator = TableGenerator(
    output_dir=model_cache_manager.get_cache_dir('models')
)

In [ ]:
from transformers import AutoModel, AutoTokenizer
bert_model_name = parameters_dict.get("bert_model_to_use", "bert-base-uncased")
bert_out_dim = AutoModel.from_pretrained(bert_model_name).config.hidden_size

# Main Method

## 1. Mount and load data

In [ ]:
drive_root_folder = mount_drive('tulane-courts-shared/v2')
court_data = load_data(drive_root_folder)

In [ ]:
model_configuration = get_model_config()

In [ ]:
court_data = append_prefix_to_proceeding_ids(court_data)
court_data = filter_proceedings(court_data)
crawled_original_data_folder_path = Path("/data/krosen5/gdrive-all/tulane-courts-shared/crawled")
court_data, multi_defendant_cases  = remove_multi_defendant_cases(court_data, crawled_original_data_folder_path)

In [ ]:
print("   Creating combined dockets...")
court_data['combined_dockets'] = create_combined_dockets(court_data)

In [ ]:
print("Removing cases that start in CDC and have no MAG proceedings...")
court_data = filter_to_magistrate_started_cases(court_data)

In [ ]:
print("Removing cases that involve some sort of extradition")
court_data = remove_extradition_cases(court_data)
court_data['combined_dockets'] = create_combined_dockets(court_data)


In [ ]:
def clean_the_dictionary(dictionary):
    elements_to_remove = model_configuration.EXCLUDED_LABELS
    cleaned_dict = {}
    
    for key, value in dictionary.items():
        if isinstance(value, list):
            # Remove unwanted elements while preserving order
            cleaned_list = [item for item in value if item not in elements_to_remove]
            # Only keep the key if the list is not empty
            if cleaned_list:
                cleaned_dict[key] = cleaned_list
        else:
            # Keep non-list values as they are
            cleaned_dict[key] = value
    
    return cleaned_dict

In [ ]:
print("2. Processing regex labels...")
labels_cache = model_cache_manager.get_cache_dir('labels')
if parameters_dict['regex_labels_mode'] == 'load_in' and parameters_dict.get('regex_labels_load_in_folder'):
    #copy from regex_folder to labels_cache the labels_cache.pkl file
    source_file = regex_folder / 'labels_cache.pkl'
    destination_file = Path(labels_cache) / 'labels_cache.pkl'
    shutil.copy2(source_file, destination_file)

cached_outcome_labels = generate_and_cache_regex_labels(court_data, labels_cache, parameters_dict['regex_labels_mode'], cache_manager=model_cache_manager)
cached_outcome_labels_v2 = clean_the_dictionary(cached_outcome_labels)    
print(f"   Original labels count: {len(cached_outcome_labels)}")
print(f"   Cleaned labels count: {len(cached_outcome_labels_v2)}")

In [ ]:
print("2b. Filtering out proceedings with zero labels...")
court_data, cached_outcome_labels_v2 = filter_zero_label_proceedings(
    court_data, 
    cached_outcome_labels_v2
)
print(f"   Final labels count after filtering: {len(cached_outcome_labels_v2)}")
court_data['combined_dockets'] = create_combined_dockets(court_data)

In [ ]:
print("Initializing Sequence Cache (building indexes)...")
# Build the cache once using the fully cleaned 'court_data'
seq_cache = SequenceCacheBuilder(court_data)
print("Cache initialization complete.")

In [ ]:
print("3. Setting up model...")
computation_device = torch.device(f"cuda:{cuda_index}" if torch.cuda.is_available() else "cpu")
print(f"   Using device: {computation_device}") 
case_outcome_model = create_model_from_data(court_data, model_cache_manager.get_cache_dir('bert'), device = computation_device, cache_manager=model_cache_manager, parameters_dict=parameters_dict, bert_out_dim=bert_out_dim)
analyze_model_memory(case_outcome_model)

In [ ]:
print_label_distribution_by_court(
    court_data,
    cached_outcome_labels_v2,
    model_configuration.LABEL_NAMES,
    mag_key="mag",
    cdc_key="cdc",
    proceeding_col="proceeding_id")

## 2. Instantiate model

In [ ]:
computation_device = torch.device(f"cuda:{cuda_index}" if torch.cuda.is_available() else "cpu")
bert_cache_directory = model_cache_manager.get_cache_dir('bert')
proceeding_index_file = bert_cache_directory / "proceeding_index.pkl"
# only build the index (and embeddings.h5) if it doesn't exist
print(f"Checking for cached BERT embeddings in {bert_cache_directory}")
if not proceeding_index_file.exists():
    model_name = parameters_dict['bert_model_to_use'] if 'bert_model_to_use' in parameters_dict else "bert-base-uncased"
    extract_and_cache_bert_embeddings_streaming(
        court_data,
        cache_dir=bert_cache_directory,
        model_name=parameters_dict['bert_model_to_use'] if 'bert_model_to_use' in parameters_dict else "bert-base-uncased",
        out_dim=bert_out_dim,
        batch_size=parameters_dict['bert_batch_size'] if 'bert_batch_size' in parameters_dict else 32,
        device=computation_device,
        force_recompute=False,
        cache_manager=model_cache_manager
    )

In [ ]:
print("Creating model from data")
case_outcome_model = create_model_from_data(court_data, bert_cache_directory, device=computation_device, cache_manager=model_cache_manager, parameters_dict=parameters_dict, bert_out_dim=bert_out_dim)

## 3. Build list of (mag_num, case_id, defendant_id) tuples

In [ ]:
print("Building list of defendant cases for training")
combined_dockets_df = court_data['combined_dockets']
defendant_case_identifiers = []
for _, row in combined_dockets_df.iterrows():
    case_identifier = (row.get('mag_num'), row.get('case_id'), row.get('defendant_id'))
    defendant_case_identifiers.append(case_identifier)
valid_defendant_case_ids = [
    case_id for case_id in defendant_case_identifiers 
    if pd.notna(case_id[2]) and pd.notna(case_id[0])
]

## 4. Sample a subset and create sequences with prior-case features


In [ ]:
print("Filtering to completed cases only for training")
completed_defendant_cases = filter_to_completed_cases(
    court_data, 
    valid_defendant_case_ids, 
    cached_outcome_labels_v2
)
print(f"Found {len(completed_defendant_cases)} completed cases for training")

In [ ]:
# Pre-filter valid cases first
print("Pre-filtering valid cases...")
_counts = build_case_counts_df(court_data, cached_outcome_labels_v2)

_df_cases = pd.DataFrame(completed_defendant_cases,
                         columns=['mag_num','case_id','defendant_id'])

_tagged = (_df_cases.merge(_counts, on=['mag_num','case_id','defendant_id'], how='left')
                    .assign(valid=lambda d: d['valid'].fillna(False)))

valid_completed_defendant_cases = [tuple(x) for x in
    _tagged.loc[_tagged['valid'], ['mag_num','case_id','defendant_id']].to_numpy()]

In [ ]:
if not parameters_dict['new_dataset_creation'] and parameters_dict['sampling_strategy'] is None:
    print(f"Sampling {min(parameters_dict['number_of_cases_in_sample'], len(valid_completed_defendant_cases))} out of {len(valid_completed_defendant_cases)} valid cases")
    set_global_seed(parameters_dict['random_seed'])  # Set global random state
    training_case_sample = random.sample(valid_completed_defendant_cases, min(parameters_dict['number_of_cases_in_sample'], len(valid_completed_defendant_cases)))
elif parameters_dict['sequence_data_mode'] == 'import':
    training_case_sample = None
else: 
    training_case_sample = valid_completed_defendant_cases

In [ ]:
models_directory_cooccurences_path = model_cache_manager.get_cache_dir('models')
models_directory_cooccurences_path.mkdir(parents=True, exist_ok=True)
cooccurrences_output_file = models_directory_cooccurences_path / "cooccurrence_matrix_valid_completed_cases.xlsx"

In [ ]:
export_cooccurrence_from_valid_completed_cases(
    valid_completed_defendant_cases=valid_completed_defendant_cases,
    court_data=court_data,
    cached_outcome_labels_v2=cached_outcome_labels_v2,
    label_names=model_configuration.LABEL_NAMES,
    output_path=cooccurrences_output_file
)

In [ ]:
clerk_cache_dir = model_cache_manager.get_cache_dir('clerk_analysis')
analyze_clerk_data(
    court_data,
    cached_outcome_labels_v2,
    valid_completed_defendant_cases, 
    output_dir = clerk_cache_dir
)

In [ ]:
if not parameters_dict['sequence_data_mode'] == 'import':
    print("Creating Enhanced Label Processor with predefined labels")
    # Initialize the label processor with your 22 predefined labels
    label_processor = EnhancedLabelProcessor(predefined_label_names=model_configuration.LABEL_NAMES)

    print("Creating training sequences with MultiLabelBinarizer support")
    # Use the new sequence creation function that returns raw string labels
    sequences, label_sequences_raw = create_enhanced_training_sequences_cached(
        training_case_sample, 
        court_data, 
        cached_outcome_labels_v2,
        skip_duplicate_labels=parameters_dict['skip_duplicate_labels'],
        cache = seq_cache,
    )

    print(f"Created {len(sequences)} sequences")
    print(f"Sample sequence length: {len(sequences[0]) if sequences else 0}")
    print(f"Sample labels: {label_sequences_raw[0][0] if label_sequences_raw else []}")

    # Fit the MultiLabelBinarizer on all your label data
    print("Fitting MultiLabelBinarizer on training data...")
    label_processor.fit(label_sequences_raw)

    # Verify the fitted labels match your expectations
    print(f"Label processor fitted with {len(label_processor.label_names)} labels:")
    print(f"Expected: {len(model_configuration.LABEL_NAMES)} labels")
    print(f"Match: {set(label_processor.label_names) == set(model_configuration.LABEL_NAMES)}")

In [ ]:
if not parameters_dict['sequence_data_mode'] == 'import':
    print("\n" + "="*80)
    print("TRUNCATING SEQUENCES AT CASE CONCLUSION")
    print("="*80)

    sequences, label_sequences_raw = filter_proceedings_to_end_of_case(
        sequences,
        label_sequences_raw
    )

    print(f"\nFinal sequences after truncation: {len(sequences)}")
    print("="*80)

In [ ]:
data_mode = parameters_dict.get('sequence_data_mode', 'current')
if data_mode == 'current':
    print("Using current sequences and labels.")
elif data_mode == 'export':
    if parameters_dict['new_dataset_creation']:
        print("Applying new_dataset_creation sampling...")
        set_global_seed(parameters_dict['random_seed'])  # Set global random state
        cases = list(zip(sequences, label_sequences_raw))
        if parameters_dict['sampling_strategy'] == 'oversample_long_sequences':
            print("Sampling strategy: oversample_long_sequences")
            sampled_cases, sampled_lengths, sampled_indices = sample_from_the_longest_cases(
                cases,
                number_of_longest_cases=5000,
                sample_size=parameters_dict['number_of_cases_in_sample']        
                )
            export_long_cases_to_xlsx(
                sampled_cases, 
                output_filepath=model_cache_manager.get_cache_dir('datasets') / "longest_case_samples.xlsx",
                court_data=court_data
            )
        elif parameters_dict['sampling_strategy'] == 'uniform':
            print("Sampling strategy: uniform")
            sampled_cases, sampled_lengths, sampled_indices = sample_uniformly(
                cases,
                sample_size=parameters_dict['number_of_cases_in_sample']
            )
            export_uniform_cases_to_xlsx(
                sampled_cases, 
                output_filepath=model_cache_manager.get_cache_dir('datasets') / "uniform_case_samples.xlsx",
                court_data=court_data
            )
        elif parameters_dict['sampling_strategy'] == 'balanced': 
            print("Sampling strategy: balanced")
            sampled_cases, sampled_lengths, sampled_indices, low_freq_labels = create_balanced_dataset(
                cases,
                low_frequency_threshold=parameters_dict['low_frequency_threshold'],
                target_size=parameters_dict['number_of_cases_in_sample']
            )
            export_balanced_cases_to_xlsx(
                sampled_cases,
                sampled_indices,
                output_filepath=model_cache_manager.get_cache_dir('datasets') / "balanced_case_samples.xlsx",
                low_freq_labels=low_freq_labels,
                court_data=court_data
            )
        else:
            print("No sampling strategy applied, using all cases.")
            sequences, label_sequences_raw = zip(*cases)
        # Set sequences and label_sequences_raw to the sampled cases
        sequences, label_sequences_raw = zip(*sampled_cases)
        sequences = list(sequences)
        label_sequences_raw = list(label_sequences_raw)
        print(f"Sampled down to {len(sequences)} sequences")
    
    case_identifiers = extract_case_identifier_from_sequences(sequences, label_sequences_raw)
    identifiers_path = model_cache_manager.get_cache_dir('datasets') / 'case_identifiers'
    save_case_identifiers(case_identifiers, identifiers_path)
    
    export_filename = 'sequences_and_labels.pkl'
    export_path = model_cache_manager.get_cache_dir('datasets') / export_filename
    
    print(f"Exporting {len(sequences)} sequences to: {export_path}")
    with open(export_path, 'wb') as file:
        pickle.dump((sequences, label_sequences_raw), file)

        indices_path = model_cache_manager.get_cache_dir('datasets') / 'sampled_indices.pkl'
    with open(indices_path, 'wb') as file:
        pickle.dump(sampled_indices, file)

    print(f"Saved sampled indices to: {indices_path}")

    
    # Also save the label processor
    label_processor_path = model_cache_manager.get_cache_dir('models') / 'label_processor.pkl'
    with open(label_processor_path, 'wb') as file:
        pickle.dump(label_processor, file)
    
    print(f"Exported {len(sequences)} sequences")
    print(f"Sequences: {export_path}")
    print(f"Label processor: {label_processor_path}")

elif data_mode == 'import':
    import_filename = 'sequences_and_labels.pkl'
    import_path = import_cache_manager.get_cache_dir('datasets') / import_filename
    
    if not import_path.exists():
        raise FileNotFoundError(f"Import file not found: {import_path}")
    
    print(f"Importing sequences from: {import_path}")
    with open(import_path, 'rb') as file:
        sequences, label_sequences_raw = pickle.load(file)

    #sampled indices loading
    sampled_indices_path = import_cache_manager.get_cache_dir('datasets') / 'sampled_indices.pkl'
    if sampled_indices_path.exists():
        with open(sampled_indices_path, 'rb') as file:
            original_sampled_indices = pickle.load(file)
        print(f"Loaded original sampled indices: {len(original_sampled_indices)} indices")
    else:
        print("Warning: Original sampled_indices not found. Cannot map back to original cases.")
        original_sampled_indices = None
    
    case_identifiers_before_skipping = extract_case_identifier_from_sequences(sequences, label_sequences_raw)
    case_identifiers_before_path = model_cache_manager.get_cache_dir('datasets') / 'case_identifiers_before_skipping'
    save_case_identifiers(case_identifiers_before_skipping, case_identifiers_before_path)
    
    # Load the label processor
    label_processor_path = import_cache_manager.get_cache_dir('models') / 'label_processor.pkl'
    if label_processor_path.exists():
        print(f"Loading label processor from: {label_processor_path}")
        with open(label_processor_path, 'rb') as file:
            label_processor = pickle.load(file)
    else:
        raise FileNotFoundError(f"Label processor not found: {label_processor_path}. Cannot proceed without it.")
    
    #make a copy of label_processor in the model cache manager
    label_processor_copy_path = model_cache_manager.get_cache_dir('models') / 'label_processor.pkl'
    with open(label_processor_copy_path, 'wb') as file:
        pickle.dump(label_processor, file)
    print(f"Copied label processor to: {label_processor_copy_path}")
    

    
    if parameters_dict['skip_duplicate_labels_on_import']:
        #use apply_skipped_duplicates_on_imported_sequences
        sequences, label_sequences_raw, kept_relative_indices = apply_skipped_duplicates_on_imported_sequences(
            sequences,
            label_sequences_raw,
            return_indices = True
        )
        if original_sampled_indices is not None:
            print("Mapping back to original cases using sampled indices...")
            kept_original_indices = [original_sampled_indices[i] for i in kept_relative_indices]
            # Optionally, save the updated sampled indices
            original_indices_path = model_cache_manager.get_cache_dir('datasets') / 'kept_original_indices.pkl'
            with open(original_indices_path, 'wb') as file:
                pickle.dump(kept_original_indices, file)
            print(f"  Saved to: {original_indices_path}")
        else:
            kept_original_indices = kept_relative_indices
            #make a warning message
            print("Warning: original_sampled_indices is None, so kept_original_indices are relative to imported data.")
        
        case_identifiers_after_skipping = extract_case_identifier_from_sequences(sequences, label_sequences_raw)
        case_identifiers_after_path = model_cache_manager.get_cache_dir('datasets') / 'case_identifiers_after_skipping'
        save_case_identifiers(case_identifiers_after_skipping, case_identifiers_after_path)

        new_import_path = model_cache_manager.get_cache_dir('datasets') / import_filename
        #redump the sequences and labels after applying skipped duplicates
        with open(new_import_path, 'wb') as file:
            pickle.dump((sequences, label_sequences_raw), file)
        #depending on the sampling strategy, we will export it to excel  
        if parameters_dict['sampling_strategy'] == 'balanced':
            export_balanced_cases_to_xlsx(
                list(zip(sequences, label_sequences_raw)),
                kept_original_indices,
                output_filepath=model_cache_manager.get_cache_dir('datasets') / "balanced_case_samples_after_skipping.xlsx",
                court_data=court_data
            )
        elif parameters_dict['sampling_strategy'] == 'uniform':
            export_uniform_cases_to_xlsx(
                list(zip(sequences, label_sequences_raw)),
                output_filepath=model_cache_manager.get_cache_dir('datasets') / "uniform_case_samples_after_skipping.xlsx",
                court_data=court_data
            )
        elif parameters_dict['sampling_strategy'] == 'oversample_long_sequences':
            export_long_cases_to_xlsx(
                list(zip(sequences, label_sequences_raw)),
                output_filepath=model_cache_manager.get_cache_dir('datasets') / "longest_case_samples_after_skipping.xlsx",
                court_data=court_data
            )
        else:
            print("No sampling strategy applied, not exporting to excel after skipping duplicates.")
    if parameters_dict['skip_duplicate_labels_on_import']:
        compare_dictionary = compare_case_identifiers(
            case_identifiers_before_skipping,
            case_identifiers_after_skipping
        )
        #export comparison results
        comparison_path = model_cache_manager.get_cache_dir('datasets') / 'case_identifier_comparison.pkl'
        with open(comparison_path, 'wb') as file:
            pickle.dump(compare_dictionary, file)
        print(f"Saved case identifier comparison to: {comparison_path}")

    print(f"Imported {len(sequences)} sequences")
    print(f"  - Sequences: {import_path}")
    print(f"  - Label processor: {label_processor_path}")


## 5. Split into train/val or k-fold cross validation  and train with teacher forcing

In [ ]:
if sequences and label_sequences_raw:
    # Filter sequences with meaningful labels using helper function
    filtered_training_data = filter_sequences_with_meaningful_labels(
        sequences, 
        label_sequences_raw
    )

    #check if filter to smaller exists within the parameters dict and if it is set to true then run the filtering
    if parameters_dict.get('filter_to_smaller', False):
        print("Filtering to smaller subset of sequences for cross-validation...")
        subset_size = parameters_dict['number_of_cases_in_sample']
        filtered_training_data = random.sample(
            filtered_training_data,
            k=min(subset_size, len(filtered_training_data))
        )
        print(f"✅ Using {len(filtered_training_data)} sequences for cross-validation subset after filtering.")
    # Map to original variable names for consistency
    sequences, label_sequences_raw = map(list, zip(*filtered_training_data))
    print(f"✅ Using {len(sequences)} sequences for cross-validation subset.")

In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        # Split into train/val using helper function
        set_global_seed(parameters_dict['random_seed'])  # Set global random state
        train_sequences_and_labels, val_sequences_and_labels = split_train_validation(
            filtered_training_data,
            test_size=0.2,
            random_state=parameters_dict['random_seed']        
            )
        # Extract sequences and labels separately using helper function
        train_sequences, train_labels_raw = prepare_sequences_and_labels(train_sequences_and_labels)
        val_sequences, val_labels_raw = prepare_sequences_and_labels(val_sequences_and_labels)  
        print(f"\nData split complete:")
        print(f"  Training: {len(train_sequences)} sequences")
        print(f"  Validation: {len(val_sequences)} sequences")
    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        print("Preparing data for {parameters_dict['k_fold_num_folds']}-fold cross-validation")
        print("Total sequences for K-Fold CV: ", len(filtered_training_data))
        # For CV, we don't split yet - we'll split during the CV process
        # But we need to define a function to create fresh models
        def create_fresh_model():
            fresh_model = create_model_from_data(data = court_data, cache_dir = model_cache_manager.get_cache_dir('bert'), device = computation_device, cache_manager=model_cache_manager, parameters_dict=parameters_dict, bert_out_dim=bert_out_dim)
            print(fresh_model)
            return fresh_model
        
        # Store the filtered data for later use in CV
        cv_full_data = filtered_training_data

In [ ]:
## 7. Generate Co-occurrence Matrices

if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        # Use helper function to convert raw labels to binary format
        train_data_binary = convert_data_to_format(
            train_sequences_and_labels,
            label_processor,
            format_type='binary'
        )
        
        val_data_binary = convert_data_to_format(
            val_sequences_and_labels,
            label_processor,
            format_type='binary'
        )
        models_directory = model_cache_manager.get_cache_dir('models')
        models_directory.mkdir(parents=True, exist_ok=True)

        # Use helper function to generate co-occurrence matrices
        generate_cooccurrence_matrix_excel(
            train_data_binary,
            label_processor,
            models_directory,
            'cooccurrence_matrix_training_mlb.xlsx',
            'training set'
        )
        
        generate_cooccurrence_matrix_excel(
            val_data_binary,
            label_processor,
            models_directory,
            'cooccurrence_matrix_validation_mlb.xlsx',
            'validation set'
        )
    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        # For K-Fold CV, convert the full filtered data to binary format
        cv_data_binary = convert_data_to_format(
            cv_full_data,
            label_processor,
            format_type='binary'
        )
        models_directory = model_cache_manager.get_cache_dir('models')
        models_directory.mkdir(parents=True, exist_ok=True)

        # Use helper function to generate co-occurrence matrix for CV data
        generate_cooccurrence_matrix_excel(
            cv_data_binary,
            label_processor,
            models_directory,
            'cooccurrence_matrix_kfold_cv_mlb.xlsx',
            'K-Fold CV set'
        )

In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        perform_frequency_analysis(
            train_sequences_and_labels, 
            val_sequences_and_labels
        )
    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        perform_frequency_analysis(
            cv_full_data,
            []  # No separate validation set in K-Fold CV
        )

In [2]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        print("Debugging model with a sample from training data...")
        debug_model_sample(train_sequences_and_labels, case_outcome_model)
        
        trained_case_model, trained_case_logger = train_model(
            case_outcome_model,
            train_sequences_and_labels,
            label_processor,
            val_sequences_and_labels,
            computation_device,
            epochs=parameters_dict['number_of_epochs_for_training'],
            batch_size=parameters_dict['batch_size_for_training'],
            use_mixed_precision=parameters_dict['use_mixed_precision'],
            use_learning_rate_scheduler=parameters_dict['use_learning_rate_scheduler'],
            gradient_clip_norm=parameters_dict['gradient_clip_norm'],
            weighted_loss=parameters_dict['weighted_loss_function'],
            fold_idx=None,
            cache_manager=model_cache_manager
        )
    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        print("\nStarting K-Fold Cross-Validation training...")
        set_global_seed(parameters_dict['random_seed'])
        cv_runner = KFoldCVRunner(
            data=filtered_training_data,
            model_creation_func=create_fresh_model,
            label_processor=label_processor,
            device=computation_device,
            n_splits=parameters_dict['k_fold_num_folds'],
            epochs=parameters_dict['number_of_epochs_for_training'],
            batch_size=parameters_dict['batch_size_for_training'],
            cache_manager=model_cache_manager,
            shuffle=parameters_dict['k_fold_shuffle'],
            use_mixed_precision=parameters_dict['use_mixed_precision'],
            use_learning_rate_scheduler=parameters_dict['use_learning_rate_scheduler'],
            gradient_clip_norm=parameters_dict['gradient_clip_norm'],
            random_state=parameters_dict['random_seed'],
            weighted_loss=parameters_dict['weighted_loss_function']
        ).run()
        
        trained_case_model = cv_runner.load_best_model()
        val_sequences_and_labels, val_data_for_eval = cv_runner.get_best_fold_val_data()

        # stash for later blocks if needed
        cv_training_results = {
            "fold_results": cv_runner.fold_results,
            "aggregate_metrics": cv_runner.aggregate_metrics,
            "best_fold_idx": cv_runner.best_fold_idx,
            "n_splits": cv_runner.n_splits,
        }
        print("K-Fold Cross-Validation training complete.")
        print(f"Best fold: {cv_training_results['best_fold_idx'] + 1} out of {cv_training_results['n_splits']}")
    print("\nTraining complete.")

NameError: name 'sequences' is not defined

In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        val_data_for_eval = prepare_validation_data_for_eval(
            val_sequences_and_labels, 
            label_processor
        )
        
        validation_metrics, predictions = evaluate_model(
            trained_case_model,
            val_data_for_eval,
            computation_device,
            label_processor,
            fold_idx=None,
            cache_manager=model_cache_manager
        )
        print_micro_averaged_metrics(validation_metrics)
        print_top_performing_labels(validation_metrics, top_n=parameters_dict['top_n_performing_labels_to_display'])
    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        cv_runner.print_evaluation_summary()
        #get validation metrics from best fold
        #evaluate each fold
        validation_metrics_per_fold = {}
        predictions_per_fold = {}
        for fold_idx in range(cv_runner.n_splits):
            val_sequences_and_labels, val_data_for_eval = cv_runner.get_fold_val_data(fold_idx)
            fold_model = cv_runner.load_fold_model(fold_idx)
            train_data_for_fold = cv_runner.get_fold_train_data(fold_idx)
            fold_validation_metrics, predictions = evaluate_model(
                fold_model,
                val_data_for_eval,
                computation_device,
                label_processor,
                fold_idx=fold_idx,
                cache_manager=model_cache_manager
            )
            validation_metrics_per_fold[fold_idx] = fold_validation_metrics
            predictions_per_fold[fold_idx] = predictions

            table_generator.add_k_fold_results(
                fold_idx,
                fold_validation_metrics,
                case_results=None,
                train_size=len(train_data_for_fold),
                val_size=len(val_sequences_and_labels)
            )
            
        print("\nValidation Metrics for Each Fold:")

In [ ]:
if sequences and label_sequences_raw:
    set_global_seed(parameters_dict['random_seed'])
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        case_results = evaluate_case_level(
            trained_case_model,
            val_sequences_and_labels,
            label_processor
        )
        
        print_case_level_metrics(case_results)
        
        print_best_worst_cases(case_results, top_n=parameters_dict['num_best_and_worst_cases_to_display'])
        print_f1_distribution(case_results)
        
        case_level_output_path = models_directory / 'case_level_evaluation.xlsx'
        save_case_level_results(case_results, case_level_output_path)
        
        case_level_label_df = calculate_label_metrics_from_cases(
            case_results, 
            label_processor
        )
        
        save_label_metrics_to_excel(
            case_level_label_df, 
            case_level_output_path
        )
        
        table_generator.add_main_model_results(
            step_metrics=validation_metrics,
            case_results=case_results,
            model_name="Main Transformer (Train/Val Split)"
        )

    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        case_level_dict = {}
        for fold_idx in range(cv_runner.n_splits):
            val_sequences_and_labels, val_data_for_eval = cv_runner.get_fold_val_data(fold_idx)
            fold_model = cv_runner.load_fold_model(fold_idx)
            fold_case_results = evaluate_case_level(
                fold_model,
                val_sequences_and_labels,
                label_processor
            )
            case_level_dict[fold_idx] = fold_case_results
            print(f"\nCase-Level Metrics for Fold {fold_idx + 1}:")
            print_case_level_metrics(fold_case_results)
            print_best_worst_cases(
                fold_case_results,
                top_n=parameters_dict['num_best_and_worst_cases_to_display']
            )
            print_f1_distribution(fold_case_results)
            # Now evaluate case-level metrcis for each fold
            case_level_output_path = models_directory / cv_runner.get_fold_filename(
                'case_level_evaluation',
                fold_idx
            )
            case_level_df = calculate_label_metrics_from_cases(
                fold_case_results,
                label_processor
            )
            save_label_metrics_to_excel(
                case_level_df,
                case_level_output_path
            )
        cv_runner.aggregate_case_level_metrics()

In [ ]:
set_global_seed(parameters_dict['random_seed'])
if sequences and label_sequences_raw:
    print("\nRunning ablation study...")
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        ablation_configs = create_standard_ablation_configurations()
        fresh_model = create_model_from_data(data=court_data, cache_dir=model_cache_manager.get_cache_dir('bert'), device=computation_device, cache_manager=model_cache_manager, parameters_dict=parameters_dict, bert_out_dim=bert_out_dim)

        ablation_results = run_ablation_study(
            base_model=fresh_model,
            train_data=train_sequences_and_labels,
            val_data=val_sequences_and_labels,
            label_processor=label_processor,
            device=computation_device,
            ablation_configs=ablation_configs,
            output_dir=model_cache_manager.get_cache_dir('ablation_study'),
            epochs=parameters_dict['number_of_epochs_for_training'],
            batch_size=parameters_dict['batch_size_for_training'],
            use_mixed_precision=parameters_dict['use_mixed_precision'],
            use_learning_rate_scheduler=parameters_dict['use_learning_rate_scheduler'],
            gradient_clip_norm=parameters_dict['gradient_clip_norm'],
            weighted_loss=parameters_dict['weighted_loss_function'],
            fold_idx=None,
            cache_manager=model_cache_manager
        )
        df = compare_ablation_table(ablation_results)

    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        ablation_configs = create_standard_ablation_configurations()
        # do it for each fold's training/validation split
        df_dict_per_fold = {}
        ablation_results_per_fold = {}

        for fold_idx in range(cv_training_results['n_splits']):
            print(f"\nRunning ablation study for fold {fold_idx + 1}...")
            val_sequences_and_labels, val_data_for_eval = cv_runner.get_fold_val_data(fold_idx)
            train_data = cv_runner.get_fold_train_data(fold_idx)
            fresh_model = create_fresh_model()  # This is already defined in the notebook
            set_global_seed(parameters_dict['random_seed'])
            ablation_results = run_ablation_study(
                base_model=fresh_model,
                train_data=train_data,
                val_data=val_sequences_and_labels,
                label_processor=label_processor,
                device=computation_device,
                ablation_configs=ablation_configs,
                output_dir=model_cache_manager.get_cache_dir('ablation_study') / f'fold_{fold_idx + 1}',
                epochs=parameters_dict['number_of_epochs_for_training'],
                batch_size=parameters_dict['batch_size_for_training'],
                use_mixed_precision=parameters_dict['use_mixed_precision'],
                use_learning_rate_scheduler=parameters_dict['use_learning_rate_scheduler'],
                gradient_clip_norm=parameters_dict['gradient_clip_norm'],
                weighted_loss=parameters_dict['weighted_loss_function'],
                fold_idx=fold_idx,
                cache_manager=model_cache_manager
            )

            # keep raw results per fold
            ablation_results_per_fold[fold_idx] = ablation_results

            df = compare_ablation_table(ablation_results)
            # add a column for fold index
            df['fold_idx'] = fold_idx
            df_dict_per_fold[fold_idx] = df

        # combine all folds into one dataframe if you want it
        combined_df = pd.concat(df_dict_per_fold.values(), ignore_index=True)


In [ ]:
if parameters_dict['model_validation_strategy'] == 'train_val_split':
    print("\nAblation study complete. Saving results...")
    df.to_excel(model_cache_manager.get_cache_dir('ablation_study') / "ablation_comparison_table.xlsx")
elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
    print("\nAblation study complete for all folds. Saving combined results...")
    combined_df.to_excel(model_cache_manager.get_cache_dir('ablation_study') / "ablation_comparison_table_kfold.xlsx")

In [ ]:
# ADD ABLATION RESULTS TO COMPARISON TABLES
if parameters_dict['model_validation_strategy'] == 'train_val_split':
    if 'ablation_results' in locals() and ablation_results:
        print("Adding ablation study results to comparison tables...")
        for config_name, ablation_result in ablation_results.items():
            ablation_config = ablation_result['ablation_config']
            ablated_features = [k for k, v in ablation_config.items() if v]

            table_generator.add_ablation_results(
                config_name=config_name,
                ablated_features=ablated_features,
                step_metrics=ablation_result['metrics'],
                case_results=None  # Ablation typically doesn't include case-level
            )
        print(f"   ✅ Added {len(ablation_results)} ablation configurations")

elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
    if 'ablation_results_per_fold' in locals() and ablation_results_per_fold:
        print("Adding ablation study results to comparison tables (K-Fold)...")
        total_configs = 0

        for fold_idx, fold_ablation_results in ablation_results_per_fold.items():
            for config_name, ablation_result in fold_ablation_results.items():
                ablation_config = ablation_result['ablation_config']
                ablated_features = [k for k, v in ablation_config.items() if v]

                # include fold info in the name so you can distinguish them later
                table_generator.add_ablation_results(
                    config_name=f"{config_name} (Fold {fold_idx + 1})",
                    ablated_features=ablated_features,
                    step_metrics=ablation_result['metrics'],
                    case_results=None
                )
                total_configs += 1

        print(f"   ✅ Added {total_configs} ablation configurations across {len(ablation_results_per_fold)} folds")


In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        set_global_seed(parameters_dict['random_seed'])
        embedding_results = run_embedding_clusters_pipeline(
            model=trained_case_model,
            data=val_sequences_and_labels,
            label_processor=label_processor,
            device=computation_device,
            output_dir=model_cache_manager.get_cache_dir('embedding_analysis'),
            max_samples=None,
            n_clusters=6,
            use_teacher_forcing=True,
            run_dbscan=False,
            show_elbow=True,
            verbose=True,
            random_state=parameters_dict['random_seed']
        )

    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        set_global_seed(parameters_dict['random_seed'])
        kfold_embedding_results = run_embedding_clusters_kfolds_pipeline(
            cv_runner=cv_runner,
            label_processor=label_processor,
            device=computation_device,
            output_dir=model_cache_manager.get_cache_dir('embedding_analysis'),
            max_samples_per_fold=None,
            n_clusters=6,
            use_teacher_forcing=True,
            run_dbscan=False,
            show_elbow=True,
            verbose=True,
            random_state=parameters_dict['random_seed']
        )

In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        # Use helper function for diagnostic analysis
        diagnostic_results = run_diagnostic_analysis(
            val_data_for_eval,
            label_processor,
            trained_case_model,
            computation_device
        )
    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        diagnostic_results = run_diagnostic_analysis(
            val_data_for_eval,  # Best fold's validation data
            label_processor,
            trained_case_model,
            computation_device
        )

In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        # Use helper function for baseline comparison
        set_global_seed(parameters_dict['random_seed'])
        baseline_model, baseline_results, baseline1_predictions_binary = run_baseline_comparison(
            train_sequences_and_labels,
            val_sequences_and_labels,
            label_processor,
            epochs=parameters_dict['number_of_epochs_for_training'],
            batch_size=parameters_dict['batch_size_for_training']
        )

    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        # do the baseline on each fold's validation data
        baseline_results_per_fold = {}
        baseline_pred_binaries_per_fold = {}
        set_global_seed(parameters_dict['random_seed'])
        for fold_idx in range(cv_training_results['n_splits']):
            print(f"Running baseline comparison on fold {fold_idx + 1} validation data.")
            fold = cv_training_results['fold_results'][fold_idx]
            baseline_model, baseline_results, baseline1_predictions_binary = run_baseline_comparison(
                fold['train_data'],
                fold['val_data'],
                label_processor,
                epochs=parameters_dict['number_of_epochs_for_training'],
                batch_size=parameters_dict['batch_size_for_training']
            )
            baseline_results_per_fold[fold_idx] = baseline_results
            baseline_pred_binaries_per_fold[fold_idx] = baseline1_predictions_binary


In [ ]:
if parameters_dict['model_validation_strategy'] == 'train_val_split':
    if 'baseline_results' in locals() and baseline_results:
        print("\n📊 Adding baseline model results to comparison tables...")
        formatted_baseline = convert_baseline_results_to_table_format(baseline_results)
        table_generator.add_baseline_results(
            step_metrics=formatted_baseline,
            case_results=None,  # no case-level for baseline
            model_name="Baseline Model (Train/Val Split)",
            type_of_model='Baseline Model',
            results_key = 'baseline_1'
        )
        print("   ✅ Added baseline model results")

elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
    if 'baseline_results_per_fold' in locals() and baseline_results_per_fold:
        print("\n📊 Adding baseline model results to comparison tables (K-Fold)...")
        total = 0

        for fold_idx, fold_baseline_results in baseline_results_per_fold.items():
            formatted_fold_baseline = convert_baseline_results_to_table_format(fold_baseline_results)
            table_generator.add_baseline_fold_results(
                step_metrics=formatted_fold_baseline,
                case_results=None,  # no case-level for baseline
                model_name=f"Baseline Model",
                fold_idx=fold_idx,
                type_of_model='Baseline Model',
                results_key = 'baseline_1_folds'
            )
            total += 1

        print(f"   ✅ Added baseline model results for {total} folds")


In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        # Use helper function for random baseline comparison
        set_global_seed(parameters_dict['random_seed'])
        baseline2_model, baseline2_results = run_random_baseline_comparison(
            train_sequences_and_labels,
            val_sequences_and_labels,
            label_processor,
            random_seed=parameters_dict['random_seed']
        )
    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        # Run random baseline on each fold's validation data
        baseline2_results_per_fold = {}
        set_global_seed(parameters_dict['random_seed'])
        for fold_idx in range(cv_training_results['n_splits']):
            print(f"Running random baseline on fold {fold_idx + 1} validation data.")
            fold = cv_training_results['fold_results'][fold_idx]
            
            baseline2_model, baseline2_results = run_random_baseline_comparison(
                fold['train_data'],
                fold['val_data'],
                label_processor,
                random_seed=parameters_dict['random_seed']
            )
            baseline2_results_per_fold[fold_idx] = baseline2_results

In [ ]:
if parameters_dict['model_validation_strategy'] == 'train_val_split':
    if 'baseline2_results' in locals() and baseline2_results:
        print("\n📊 Adding random baseline results to comparison tables...")
        table_generator.add_baseline_results(
            step_metrics=baseline2_results,
            case_results=None,  # no case-level for baseline
            model_name="Random Baseline (Frequency-based)", 
            type_of_model='Random Baseline',
            results_key='baseline_2'
        )
        print("   ✅ Added random baseline results")

elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
    if 'baseline2_results_per_fold' in locals() and baseline2_results_per_fold:
        print("\n📊 Adding random baseline results to comparison tables (K-Fold)...")
        total = 0
        
        for fold_idx, fold_baseline2_results in baseline2_results_per_fold.items():
            table_generator.add_baseline_fold_results(
                step_metrics=fold_baseline2_results,
                case_results=None,  # no case-level for baseline
                model_name="Random Baseline",
                fold_idx=fold_idx,
                type_of_model='Random Baseline',
                results_key='baseline_2_folds'
            )
            total += 1
        
        print(f"   ✅ Added random baseline results for {total} folds")

In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        # use the baselien 1 predictions
        baseline_predictions_binary = baseline1_predictions_binary
        #get the predictions from evaluate_model
        transformer_predictions_binary = predictions
        #use the compare_transformer_vs_baseline1_on_completion_labels function
        comparison_results = compare_transformer_vs_baseline1_on_completion_labels(
            transformer_predictions_binary,
            baseline_predictions_binary,
            val_sequences_and_labels,
            label_processor
        )
        #now convert to table generator format using convert_baseline1_comparison_to_table_generator_format
        transformer_step_metrics, baseline_step_metrics = convert_baseline1_comparison_to_table_generator_format(
            comparison_results
        )
    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        transformer_step_metrics_per_fold = {}
        baseline_step_metrics_per_fold = {}
        for fold_idx in range(cv_training_results['n_splits']):
            print(f"Comparing Transformer vs Baseline 1 on fold {fold_idx + 1}...")
            #get baseline 1 predictions
            baseline_predictions_binary = baseline_pred_binaries_per_fold[fold_idx]
            #get transformer predictions
            transformer_predictions_binary = predictions_per_fold[fold_idx]
            fold = cv_training_results['fold_results'][fold_idx]
            val_sequences_and_labels = fold['val_data']
            #compare
            comparison_results = compare_transformer_vs_baseline1_on_completion_labels(
                transformer_predictions_binary,
                baseline_predictions_binary,
                val_sequences_and_labels,
                label_processor
            )
            #convert to table generator format
            transformer_step_metrics, baseline_step_metrics = convert_baseline1_comparison_to_table_generator_format(
                comparison_results
            )
            transformer_step_metrics_per_fold[fold_idx] = transformer_step_metrics
            baseline_step_metrics_per_fold[fold_idx] = baseline_step_metrics

In [ ]:
if parameters_dict['model_validation_strategy'] == 'train_val_split':
    if ('transformer_step_metrics' in locals() and transformer_step_metrics) and ('baseline_step_metrics' in locals() and baseline_step_metrics):
        print("\n📊 Adding baseline model results to comparison tables...")
        #first add transformer only
        table_generator.add_baseline_results(
            step_metrics=transformer_step_metrics,
            case_results=None,  # no case-level for baseline
            model_name="Transformer Only Completion (Train/Val Split)",
            type_of_model='Transformer Model Only Completion',
            results_key = 'transformer_only_completion'
        )
        #then add baseline 1 only
        table_generator.add_baseline_results(
            step_metrics=baseline_step_metrics,
            case_results=None,  # no case-level for baseline
            model_name="Baseline 1 Only Completion (Train/Val Split)",
            type_of_model='Baseline Model 1 Only Completion',
            results_key = 'baseline_1_only_completion'
        )
        print("   ✅ Added baseline model results")

elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
    if ('transformer_step_metrics_per_fold' in locals() and transformer_step_metrics_per_fold) and ('baseline_step_metrics_per_fold' in locals() and baseline_step_metrics_per_fold):
        print("\n📊 Adding baseline model results to comparison tables (K-Fold)...")
        for fold_idx in range(cv_training_results['n_splits']):
            #first add transformer only
            table_generator.add_baseline_fold_results(
                step_metrics=transformer_step_metrics_per_fold[fold_idx],
                case_results=None,  # no case-level for baseline
                model_name="Transformer Only Completion",
                fold_idx=fold_idx,
                type_of_model='Transformer Model Only Completion',
                results_key = 'transformer_only_completion_folds'
            )
            #then add baseline 1 only
            table_generator.add_baseline_fold_results(
                step_metrics=baseline_step_metrics_per_fold[fold_idx],
                case_results=None,  # no case-level for baseline
                model_name="Baseline 1 Only Completion",
                fold_idx=fold_idx,
                type_of_model='Baseline Model 1 Only Completion',
                results_key = 'baseline_1_only_completion_folds'
            )

In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        # Create case IDs for validation sequences
        val_case_ids = []
        for seq in val_sequences:
            case_id = extract_case_info_from_sequence(seq, court_data)
            val_case_ids.append(case_id)
        
        interpretability_results = run_interpretability_analysis(
            trained_case_model,
            label_processor,
            device,
            val_sequences,
            val_labels_raw,
            val_case_ids,
            court_data,
            model_configuration,
            cache_manager=model_cache_manager,
            num_samples=parameters_dict['numbers_of_cases_to_analyze_for_interpretability']
        )
    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        print(f"Running interpretability analysis on each fold")
        intepretability_results_per_fold = {}
        for fold_idx in range(cv_training_results['n_splits']):
            print(f"  Fold {fold_idx + 1}...")
            fold = cv_training_results['fold_results'][fold_idx]
            fold_val_data = fold['val_data']
            
            fold_val_sequences, fold_val_labels_raw = prepare_sequences_and_labels(
                fold_val_data
            )
            
            # Create case IDs
            val_case_ids = []
            for seq in fold_val_sequences:
                case_id = extract_case_info_from_sequence(seq, court_data)
                val_case_ids.append(case_id)
            
            interpretability_results = run_interpretability_analysis(
                cv_runner.load_fold_model(fold_idx),
                label_processor,
                device,
                fold_val_sequences,
                fold_val_labels_raw,
                val_case_ids,
                court_data,
                model_configuration,
                cache_manager=model_cache_manager,
                num_samples=parameters_dict['numbers_of_cases_to_analyze_for_interpretability']
            )
            intepretability_results_per_fold[fold_idx] = interpretability_results

In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        interpreter = interpretability_results['interpreter']
        
        magnitude_results = interpreter.run_counterfactual_magnitude_study(
            val_sequences=val_sequences,
            val_labels=val_labels_raw,
            output_dir=model_cache_manager.get_cache_dir('interpretability_results') / 'magnitude_study',
            max_samples=len(val_sequences),
        )

    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        # Pool all fold val sets — each case appears in exactly one fold's val set
        all_val_sequences = []
        all_val_labels    = []

        for fold_idx in range(cv_training_results['n_splits']):
            fold_val_data = cv_training_results['fold_results'][fold_idx]['val_data']
            fold_seqs, fold_labels = prepare_sequences_and_labels(fold_val_data)
            all_val_sequences.extend(fold_seqs)
            all_val_labels.extend(fold_labels)

        # Use the best fold's interpreter
        best_fold_idx   = cv_training_results['best_fold_idx']
        best_interpreter = intepretability_results_per_fold[best_fold_idx]['interpreter']

        magnitude_results = best_interpreter.run_counterfactual_magnitude_study(
            val_sequences=all_val_sequences,
            val_labels=all_val_labels,
            output_dir=model_cache_manager.get_cache_dir('interpretability_results') / 'magnitude_study',
            max_samples=len(all_val_sequences),
        )

In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        # get the interpreter from prior results (run_interpretability_analysis)
        interpreter = interpretability_results['interpreter']  # fixed key
        val_pairs_binary = [
            (seq, label_processor.transform_sequence(labels))
            for (seq, labels) in val_sequences_and_labels
            if len(seq) >= 1
        ]

        attributions_tables = compute_global_attributions_table(
            interpreter,
            val_pairs_binary,
            max_samples=200,
            show_top_n_features=20,
            ig_n_steps=200,
            optional_tag = None,
            verbose=True,
            interpretability_folder=model_cache_manager.get_cache_dir('interpretability_results')
        )

        attributions_tables['full'].to_excel(
            model_cache_manager.get_cache_dir('interpretability_results') / 'attributions_complete_all_views.xlsx',
            index=True
        )
        attributions_tables['raw'].to_excel(
            model_cache_manager.get_cache_dir('interpretability_results') / 'table1_raw_attributions.xlsx',
            index=True
        )
        
        attributions_tables['within_label'].to_excel(
            model_cache_manager.get_cache_dir('interpretability_results') / 'table2_within_label_zscores.xlsx',
            index=True
        )

        attributions_tables['across_label'].to_excel(
            model_cache_manager.get_cache_dir('interpretability_results') / 'table3_across_label_zscores.xlsx',
            index=True
        )

    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        attributions_tables_per_fold = {}
        for fold_idx in range(cv_training_results['n_splits']):
            print(f"Computing global attributions table for fold {fold_idx + 1}...")

            fold_interpretability_results = intepretability_results_per_fold[fold_idx]
            interpreter = fold_interpretability_results['interpreter']
            # get val data for this fold
            fold = cv_training_results['fold_results'][fold_idx]
            val_pairs_binary = [
                (seq, label_processor.transform_sequence(labels))
                for (seq, labels) in fold['val_data']
                if len(seq) >= 1
            ]

            attributions_tables = compute_global_attributions_table(
                interpreter,
                val_pairs_binary,
                max_samples=200,
                show_top_n_features=20,
                ig_n_steps=200,
                optional_tag = f'fold_{fold_idx + 1}',
                verbose=True,
                interpretability_folder=model_cache_manager.get_cache_dir('interpretability_results')
            )

            attributions_tables_per_fold[fold_idx] = attributions_tables

            # save each table per fold
            attributions_tables['full'].to_excel(
                model_cache_manager.get_cache_dir('interpretability_results') / f'attributions_complete_all_views_fold_{fold_idx + 1}.xlsx',
                index=True
            )
            attributions_tables['raw'].to_excel(
                model_cache_manager.get_cache_dir('interpretability_results') / f'table1_raw_attributions_fold_{fold_idx + 1}.xlsx',
                index=True
            )
            attributions_tables['within_label'].to_excel(
                model_cache_manager.get_cache_dir('interpretability_results') / f'table2_within_label_zscores_fold_{fold_idx + 1}.xlsx',
                index=True
            )
            attributions_tables['across_label'].to_excel(
                model_cache_manager.get_cache_dir('interpretability_results') / f'table3_across_label_zscores_fold_{fold_idx + 1}.xlsx',
                index=True
            )

In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        confident_misclass_results = run_confident_misclassification(
            trained_case_model,
            label_processor,
            computation_device,
            val_sequences_and_labels,
            cache_manager=model_cache_manager,
            threshold=parameters_dict['minimum_confidence_for_wrong_prediction_inclusion'],
            top_k=parameters_dict['number_of_highest_probability_predictions_per_step'],
            top_n=parameters_dict['maximum_number_of_confident_mistakes_to_return'],
            max_sequences=parameters_dict['maximum_validation_sequences_to_process_for_analysis_of_incorrect']
        )
        
        mistakes_report = confident_misclass_results['mistakes_report']
        interpreted_mistakes = confident_misclass_results['interpreted_mistakes']
        case_patterns = confident_misclass_results['case_patterns']
    
    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        print(f"Running confident misclassification analysis on each fold")
        confident_misclass_results_per_fold = {}
        for fold_idx in range(cv_training_results['n_splits']):
            print(f"  Fold {fold_idx + 1}...")
            fold = cv_training_results['fold_results'][fold_idx]
            fold_val_data = fold['val_data']
            
            fold_confident_misclass_results = run_confident_misclassification(
                cv_runner.load_fold_model(fold_idx),
                label_processor,
                computation_device,
                fold_val_data,
                cache_manager=model_cache_manager,
                threshold=parameters_dict['minimum_confidence_for_wrong_prediction_inclusion'],
                top_k=parameters_dict['number_of_highest_probability_predictions_per_step'],
                top_n=parameters_dict['maximum_number_of_confident_mistakes_to_return'],
                max_sequences=parameters_dict['maximum_validation_sequences_to_process_for_analysis_of_incorrect']
            )
            confident_misclass_results_per_fold[fold_idx] = fold_confident_misclass_results

In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        label_mistakes = analyze_confident_misclass_for_label(
            confident_misclass_results, 
            target_label="CHARGES_REFUSED"
        )
    
    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        # Analyze for each fold
        label_mistakes_per_fold = {}
        for fold_idx in range(cv_training_results['n_splits']):
            print(f"  Analyzing CHARGES_REFUSED mistakes for Fold {fold_idx + 1}...")
            label_mistakes_per_fold[fold_idx] = analyze_confident_misclass_for_label(
                confident_misclass_results_per_fold[fold_idx], 
                target_label="CHARGES_REFUSED"
            )

In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        analyze_sample_predictions(
            trained_case_model,
            val_sequences_and_labels,
            label_processor,
            validation_metrics,
            court_data
        )
    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        best_fold = cv_training_results['fold_results'][cv_training_results['best_fold_idx']]
        analyze_sample_predictions(
            cv_runner.load_fold_model(cv_training_results['best_fold_idx']),
            best_fold['val_data'],
            label_processor,
            best_fold['metrics'], 
            court_data
        )
    

In [ ]:
if not sequences or not label_sequences_raw:
    print("❌ No sequences available for training with MultiLabelBinarizer.")

In [ ]:
#put the dictionaries into a single dictionary of dictionaries
model_results = {
    "cv_training_results": cv_training_results if 'cv_training_results' in locals() else None,
    'validation_metrics_per_fold': validation_metrics_per_fold if 'validation_metrics_per_fold' in locals() else None,
    'case_level_dict': case_level_dict if 'case_level_dict' in locals() else None,
    'ablation_results_per_fold': df_dict_per_fold if 'df_dict_per_fold' in locals() else None,
    'interpretability_results_per_fold': intepretability_results_per_fold if 'intepretability_results_per_fold' in locals() else None,
    'confident_misclass_results_per_fold': confident_misclass_results_per_fold if 'confident_misclass_results_per_fold' in locals() else None
}


In [ ]:
# write it to a file so we can get the results of it

with open(model_cache_manager.get_cache_dir('models') / 'model_results.pkl', 'wb') as file:
    pickle.dump(model_results, file)

In [ ]:
# Determine which model to export
if parameters_dict['model_validation_strategy'] == 'train_val_split':
    model_to_export = trained_case_model
    model_name = "final_model_mlb"
    
elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
    best_fold_idx = cv_training_results['best_fold_idx']
    model_to_export = cv_runner.load_fold_model(best_fold_idx)
    model_name = f"final_model_mlb_best_fold_{best_fold_idx + 1}"

# Export everything
exported_files = export_model_complete(
    model=model_to_export,
    label_processor=label_processor,
    model_name=model_name,
    cache_manager=model_cache_manager
)

In [ ]:
table1_generalized = table_generator.generate_table_1_generalized(
    baseline_1_type_of_model='Baseline Model',
    baseline_2_type_of_model='Random Baseline'
)

    
# Generate Table 2: Per-Label Detailed
table2_per_label = table_generator.generate_table_2_per_label(
    label_names=label_processor.label_names,
    baseline_1_type_of_model='Baseline Model',
    baseline_2_type_of_model='Random Baseline'
)
output_path = table_generator.save_to_excel(
    table1_df=table1_generalized,
    table2_df=table2_per_label,
    filename="model_comparison_tables.xlsx"
)